# 02 — Pré-processamento e Preparação dos Dados
## Brasileirão Série A 2025 — Previsão de Rebaixamento

**Aluno:** Leonardo Feitosa | **UFPB — Ciência de Dados**

---

## Sobre a Separação Temporal de Dados

Em projetos com **séries temporais**, a divisão entre treino e teste **não deve ser aleatória**. Utilizar uma separação aleatória (como o `train_test_split` padrão do scikit-learn) em dados temporais causa o chamado **data leakage** (vazamento de dados).

### O que é data leakage?

> **Data leakage** ocorre quando o modelo tem acesso, durante o treinamento, a informações que não estariam disponíveis no momento real da previsão.

No contexto deste projeto:
- Se utilizássemos dados de 2023 no treino, o modelo aprenderia padrões de clubes cujo resultado só seria conhecido **no futuro**.
- Isso inflacionaria artificialmente as métricas de desempenho e produziria um modelo **não generalizável** para temporadas reais futuras.

### Estratégia adotada: corte temporal

| Conjunto | Período | Uso |
|---|---|---|
| **Treino** | 2014 – 2022 | Ajuste dos parâmetros do modelo |
| **Teste** | 2023 – 2024 | Avaliação do desempenho com dados não vistos |
| **Previsão** | 2025 | Aplicação final do modelo (sem rótulo) |

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configurações visuais
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Carregamento dos dados
df = pd.read_excel(os.path.join('..', 'dados', 'BASE_FINAL.xlsx'), sheet_name='CLUBES')
df.columns = df.columns.str.strip()

# Criação da variável dependente binária
df['Status_bin'] = df['Situacao'].apply(lambda x: 1 if str(x).strip().lower() == 'rebaixado' else 0)

print(f'Base carregada: {len(df)} registros')
print(f'Temporadas disponíveis: {sorted(df["Temporada"].unique())}')
print(df[['Clube', 'Temporada', 'Plantel', 'Estrangeiros', 'Valor de Mercado Total', 'Status_bin']].head())

Base carregada: 240 registros
Temporadas disponíveis: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
              Clube  Temporada  Plantel  Estrangeiros  Valor de Mercado Total  \
0         Sao Paulo       2014       45             6                  149.65   
1       Corinthians       2014       44             6                   87.55   
2            Santos       2014       49             6                   73.40   
3  Atletico Mineiro       2014       55             2                   63.85   
4        Fluminense       2014       43             2                   48.80   

   Status_bin  
0           0  
1           0  
2           0  
3           0  
4           0  


In [3]:
# ── 1. Carregar dados de desempenho histórico (2014-2024) ──────────────────
df_desemp = pd.read_excel(
    os.path.join('..', 'dados', 'tabela_desempenho_brasileirao.xlsx'),
    sheet_name='Todos'
)
df_desemp.columns = df_desemp.columns.str.strip()
df_desemp = df_desemp.sort_values(['Clube', 'Temporada']).reset_index(drop=True)
temporadas = sorted(df_desemp['Temporada'].unique())
print(f"Desempenho carregado: {df_desemp.shape} | Temporadas: {temporadas}")
print(df_desemp.head())

Desempenho carregado: (220, 12) | Temporadas: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
   Temporada  Posicao            Clube  Jogos   V   E   D  Gols_Pro  \
0       2016       20  America Mineiro     38   7   7  24        23   
1       2018       18  America Mineiro     38  10  10  18        30   
2       2021        8  America Mineiro     38  13  14  11        41   
3       2022       10  America Mineiro     38  15   8  15        40   
4       2023       20  America Mineiro     38   5   9  24        42   

   Gols_Contra  SG  Pts  Aproveitamento  
0           58 -35   28            24.6  
1           47 -17   40            35.1  
2           37   4   53            46.5  
3           40   0   53            46.5  
4           81 -39   24            21.1  


## Janelas Deslizantes (*Sliding Windows*) — Engenharia de Características

Além das features estáticas de elenco (Plantel, Estrangeiros, Valor de Mercado), incorporamos o **histórico de desempenho recente** de cada clube, usando médias móveis de **3 e 5 temporadas anteriores**.

### Por que janelas deslizantes?

O modelo logístico não enxerga a ordem do tempo — ele trata cada temporada como uma observação independente. As janelas deslizantes *injetam* memória temporal nas features: ao saber que um clube fez em média 45 pontos nas últimas 3 temporadas, o modelo passa a capturar a **tendência de desempenho** do clube antes mesmo do campeonato começar.

### Regra anti-vazamento de dados (*anti data leakage*)

> Para a temporada **T**, a janela deslizante usa **exclusivamente** dados das temporadas **T-1, T-2, …, T-W**.  
> Nenhuma informação da temporada corrente é utilizada.

| Feature | Descrição |
|---|---|
| `Pts_media_3` | Média de pontos nas últimas 3 temporadas |
| `Pts_media_5` | Média de pontos nas últimas 5 temporadas |
| `SG_media_3` | Média de saldo de gols nas últimas 3 temporadas |
| `SG_media_5` | Média de saldo de gols nas últimas 5 temporadas |
| `Gols_Pro_media_3` | Média de gols marcados nas últimas 3 temporadas |
| `Gols_Contra_media_3` | Média de gols sofridos nas últimas 3 temporadas |
| `V_media_3` | Média de vitórias nas últimas 3 temporadas |
| `Aprv_media_3` | Média de aproveitamento nas últimas 3 temporadas |

> Clubes com menos de W temporadas anteriores recebem a média do histórico disponível (`min_periods=1`).

In [4]:
# ── 2. Calcular janelas deslizantes por clube ─────────────────────────────────
#
# shift(1): desloca 1 posição para que a média da temporada T
#           use apenas T-1, T-2, ... (sem T) → evita data leakage
#
METRICAS  = ['Pts', 'SG', 'Gols_Pro', 'Gols_Contra', 'V', 'Aproveitamento']
JANELAS   = [3, 5]

for metrica in METRICAS:
    for w in JANELAS:
        col = f'{metrica}_media_{w}'
        df_desemp[col] = (
            df_desemp.groupby('Clube')[metrica]
            .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).mean())
        )

COLS_JANELA = [f'{m}_media_{w}' for m in METRICAS for w in JANELAS]
print(f'{len(COLS_JANELA)} novas features criadas:')
print(COLS_JANELA)

# Verificação: a média da temporada T deve usar dados ANTERIORES a T
exemplo = df_desemp[df_desemp['Clube'] == 'Flamengo'][
    ['Clube', 'Temporada', 'Pts', 'Pts_media_3', 'Pts_media_5']
]
print('\nExemplo Flamengo — Pts e médias moveis:')
print(exemplo.to_string(index=False))

12 novas features criadas:
['Pts_media_3', 'Pts_media_5', 'SG_media_3', 'SG_media_5', 'Gols_Pro_media_3', 'Gols_Pro_media_5', 'Gols_Contra_media_3', 'Gols_Contra_media_5', 'V_media_3', 'V_media_5', 'Aproveitamento_media_3', 'Aproveitamento_media_5']

Exemplo Flamengo — Pts e médias moveis:
   Clube  Temporada  Pts  Pts_media_3  Pts_media_5
Flamengo       2014   52          NaN          NaN
Flamengo       2015   49    52.000000    52.000000
Flamengo       2016   71    50.500000    50.500000
Flamengo       2017   56    57.333333    57.333333
Flamengo       2018   72    58.666667    57.000000
Flamengo       2019   90    66.333333    60.000000
Flamengo       2020   71    72.666667    67.600000
Flamengo       2021   71    77.666667    72.000000
Flamengo       2022   62    77.333333    72.000000
Flamengo       2023   66    68.000000    73.200000
Flamengo       2024   70    66.333333    72.000000


In [5]:
# ── 3. Estender janelas para 2025 (previsão) ─────────────────────────────────
#
# O desempenho só existe até 2024. Para os 20 clubes da temporada 2025,
# a janela deslizante é: média dos últimos W anos ANTES de 2025,
# ou seja, usando as temporadas 2024, 2023, 2022 (janela 3) e
# 2024, 2023, 2022, 2021, 2020 (janela 5) de cada clube.
#
clubes_2025 = df['Clube'][df['Temporada'] == 2025].unique()
rows_2025   = []

for clube in clubes_2025:
    hist = df_desemp[df_desemp['Clube'] == clube].sort_values('Temporada', ascending=False)
    row  = {'Clube': clube, 'Temporada': 2025}
    for metrica in METRICAS:
        for w in JANELAS:
            ultimos = hist.head(w)[metrica]
            row[f'{metrica}_media_{w}'] = ultimos.mean() if len(ultimos) > 0 else None
    rows_2025.append(row)

df_2025_janelas = pd.DataFrame(rows_2025)

# Concatenar desempenho (2014-2024) + extensão 2025 e selecionar apenas as colunas necessárias
COLS_MERGE = ['Clube', 'Temporada'] + COLS_JANELA
df_desemp_ext = pd.concat(
    [df_desemp[COLS_MERGE], df_2025_janelas[COLS_MERGE]],
    ignore_index=True
)

print(f'Desempenho com extensão 2025: {df_desemp_ext.shape}')
print(df_2025_janelas[['Clube', 'Pts_media_3', 'Pts_media_5', 'SG_media_3']].to_string(index=False))

Desempenho com extensão 2025: (240, 14)
           Clube  Pts_media_3  Pts_media_5  SG_media_3
        Flamengo    66.000000    68.000000   18.000000
       Palmeiras    74.666667    69.600000   32.333333
        Botafogo    65.333333    53.200000   16.333333
          Santos    46.666667    53.600000   -9.000000
Atletico Mineiro    57.000000    64.600000    7.000000
          Gremio    52.000000    56.000000   -2.000000
   Vasco da Gama    45.333333    45.600000  -14.000000
        Cruzeiro    45.000000    49.000000   -4.666667
     Corinthians    57.000000    55.800000    5.333333
       Sao Paulo    55.333333    56.000000    8.333333
           Bahia    46.666667    46.600000   -4.000000
      Fluminense    57.333333    58.000000    6.666667
      Bragantino    50.000000    51.800000    0.000000
   Internacional    64.333333    62.200000   15.000000
       Fortaleza    59.000000    55.200000    7.333333
    Sport Recife    40.666667    42.800000  -18.000000
         Vitoria    42.33

In [6]:
# ── 4. Unir janelas deslizantes com a base principal ────────────────────────
df = df.merge(df_desemp_ext[COLS_MERGE], on=['Clube', 'Temporada'], how='left')
print(f"Base com janelas: {df.shape}")
print("NaN por coluna de janela:")
print(df[COLS_JANELA].isna().sum().to_string())

Base com janelas: (240, 22)
NaN por coluna de janela:
Pts_media_3               35
Pts_media_5               35
SG_media_3                35
SG_media_5                35
Gols_Pro_media_3          35
Gols_Pro_media_5          35
Gols_Contra_media_3       35
Gols_Contra_media_5       35
V_media_3                 35
V_media_5                 35
Aproveitamento_media_3    35
Aproveitamento_media_5    35


In [7]:
ANO_CORTE = 2022
ANO_PREV  = 2025
TARGET    = 'Status_bin'

# Features estáticas de elenco (Transfermarkt)
FEATURES_ELENCO = ['Plantel', 'Estrangeiros', 'Valor de Mercado Total']

# Features de janela deslizante (desempenho histórico)
FEATURES_JANELA = COLS_JANELA   # 12 colunas: Pts/SG/Gols_Pro/Gols_Contra/V/Aprv x janela 3 e 5

# Conjunto completo de features
FEATURES = FEATURES_ELENCO + FEATURES_JANELA

print(f"Total de features: {len(FEATURES)}")
print(f"  Elenco ({len(FEATURES_ELENCO)}): {FEATURES_ELENCO}")
print(f"  Janelas ({len(FEATURES_JANELA)}): {FEATURES_JANELA}")

# Separação temporal (SEM embaralhamento — série temporal)
df_rot   = df[df['Temporada'] < ANO_PREV].copy()
df_train = df_rot[df_rot['Temporada'] <= ANO_CORTE]
df_test  = df_rot[df_rot['Temporada']  > ANO_CORTE]
df_prev  = df[df['Temporada'] == ANO_PREV]

# Preencher NaN nas janelas com a mediana do treino (clubes sem historico suficiente)
mediana_treino = df_train[FEATURES_JANELA].median()
df_train = df_train.copy()
df_test  = df_test.copy()
df_prev  = df_prev.copy()
for col in FEATURES_JANELA:
    df_train[col] = df_train[col].fillna(mediana_treino[col])
    df_test[col]  = df_test[col].fillna(mediana_treino[col])
    df_prev[col]  = df_prev[col].fillna(mediana_treino[col])

print('\n' + '='*55)
print(f"  Treino  (2014-{ANO_CORTE}): {len(df_train):>4} registros | Rebaixados: {(df_train[TARGET]==1).sum()}")
print(f"  Teste   ({ANO_CORTE+1}-{ANO_PREV-1}): {len(df_test):>4} registros | Rebaixados: {(df_test[TARGET]==1).sum()}")
print(f"  Previsao ({ANO_PREV}):   {len(df_prev):>4} clubes")
print('='*55)
assert df_train['Temporada'].max() < df_test['Temporada'].min(), 'ERRO: sobreposicao temporal!'
print('\nVerificacao: sem sobreposicao entre treino e teste. OK')

Total de features: 15
  Elenco (3): ['Plantel', 'Estrangeiros', 'Valor de Mercado Total']
  Janelas (12): ['Pts_media_3', 'Pts_media_5', 'SG_media_3', 'SG_media_5', 'Gols_Pro_media_3', 'Gols_Pro_media_5', 'Gols_Contra_media_3', 'Gols_Contra_media_5', 'V_media_3', 'V_media_5', 'Aproveitamento_media_3', 'Aproveitamento_media_5']

  Treino  (2014-2022):  180 registros | Rebaixados: 36
  Teste   (2023-2024):   40 registros | Rebaixados: 8
  Previsao (2025):     20 clubes

Verificacao: sem sobreposicao entre treino e teste. OK


In [8]:
from sklearn.preprocessing import StandardScaler

# Scaler ajustado APENAS no treino — aplicado no teste e previsão sem re-ajuste
scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(df_train[FEATURES])
X_teste_sc  = scaler.transform(df_test[FEATURES])
X_prev_sc   = scaler.transform(df_prev[FEATURES])

y_treino = df_train[TARGET].values
y_teste  = df_test[TARGET].values

print(f'--- Antes da normalização (treino) | {len(FEATURES)} features ---')
print(df_train[FEATURES].describe().round(2).to_string())

print(f'\n--- Depois da normalização (treino) | media~0, std~1 ---')
print(pd.DataFrame(X_treino_sc, columns=FEATURES).describe().round(3).to_string())

--- Antes da normalização (treino) | 15 features ---
       Plantel  Estrangeiros  Valor de Mercado Total  Pts_media_3  Pts_media_5  SG_media_3  SG_media_5  Gols_Pro_media_3  Gols_Pro_media_5  Gols_Contra_media_3  Gols_Contra_media_5  V_media_3  V_media_5  Aproveitamento_media_3  Aproveitamento_media_5
count   180.00        180.00                  180.00       180.00       180.00      180.00      180.00            180.00            180.00               180.00               180.00     180.00     180.00                  180.00                  180.00
mean     55.81          4.72                   48.31        53.66        53.57        2.22        2.09             45.01             45.04                43.04                43.12      14.60      14.65                   47.09                   47.01
std       8.13          2.59                   32.74         9.05         8.57       12.08       11.27              8.07              7.50                 6.11                 5.73       3.11   

## Dados Prontos para Modelagem

Todos os conjuntos foram preparados com as features enriquecidas pelas janelas deslizantes.

| Variável | Conteúdo | Shape |
|---|---|---|
| `X_treino_sc` | Features normalizadas — treino (2014–2022) | (180, 15) |
| `X_teste_sc` | Features normalizadas — teste (2023–2024) | (40, 15) |
| `X_prev_sc` | Features normalizadas — previsão 2025 | (20, 15) |
| `y_treino` | Rótulos — treino | (180,) |
| `y_teste` | Rótulos — teste | (40,) |

**Features (15 no total):**
- **3 de elenco:** `Plantel`, `Estrangeiros`, `Valor de Mercado Total`
- **12 de janela deslizante:** `Pts`, `SG`, `Gols_Pro`, `Gols_Contra`, `V`, `Aproveitamento` × janelas de **3 e 5** temporadas

A próxima etapa é o **treinamento e avaliação dos modelos** com estas features enriquecidas.

In [9]:
import numpy as np

print('Dimensoes dos conjuntos preparados:')
print(f'  X_treino_sc : {X_treino_sc.shape}  ({len(FEATURES)} features)')
print(f'  y_treino    : {y_treino.shape} | Rebaixados: {(y_treino==1).sum()} | Permaneceram: {(y_treino==0).sum()}')
print(f'  X_teste_sc  : {X_teste_sc.shape}')
print(f'  y_teste     : {y_teste.shape} | Rebaixados: {(y_teste==1).sum()} | Permaneceram: {(y_teste==0).sum()}')
print(f'  X_prev_sc   : {X_prev_sc.shape}  (sem rótulo — temporada 2025)')

print('\nFeatures e estatísticas do scaler (media e std ajustados no treino):')
for feat, mean, std in zip(FEATURES, scaler.mean_, scaler.scale_):
    print(f'  {feat:<30} media = {mean:>10.3f} | std = {std:>10.3f}')

print('\nTodos os conjuntos estao prontos para modelagem com janelas deslizantes.')

Dimensoes dos conjuntos preparados:
  X_treino_sc : (180, 15)  (15 features)
  y_treino    : (180,) | Rebaixados: 36 | Permaneceram: 144
  X_teste_sc  : (40, 15)
  y_teste     : (40,) | Rebaixados: 8 | Permaneceram: 32
  X_prev_sc   : (20, 15)  (sem rótulo — temporada 2025)

Features e estatísticas do scaler (media e std ajustados no treino):
  Plantel                        media =     55.806 | std =      8.108
  Estrangeiros                   media =      4.717 | std =      2.585
  Valor de Mercado Total         media =     48.309 | std =     32.648
  Pts_media_3                    media =     53.661 | std =      9.027
  Pts_media_5                    media =     53.572 | std =      8.550
  SG_media_3                     media =      2.219 | std =     12.049
  SG_media_5                     media =      2.085 | std =     11.241
  Gols_Pro_media_3               media =     45.009 | std =      8.043
  Gols_Pro_media_5               media =     45.044 | std =      7.475
  Gols_Contra_me

## Justificativa das Features

O modelo utiliza **15 features** no total, combinando dados estáticos de elenco com o histórico de desempenho recente:

### Features de elenco (3)

| Feature | Descrição |
|---|---|
| `Plantel` | Tamanho total do elenco |
| `Estrangeiros` | Número de jogadores não brasileiros |
| `Valor de Mercado Total` | Soma do valor de mercado do plantel (€) |

> `ø Valor de Mercado` foi excluído por ser altamente correlacionado com `Valor de Mercado Total` (multicolinearidade sem ganho preditivo).

### Features de janela deslizante (12)

| Métrica | Janela 3 temporadas | Janela 5 temporadas |
|---|---|---|
| Pontos | `Pts_media_3` | `Pts_media_5` |
| Saldo de gols | `SG_media_3` | `SG_media_5` |
| Gols marcados | `Gols_Pro_media_3` | `Gols_Pro_media_5` |
| Gols sofridos | `Gols_Contra_media_3` | `Gols_Contra_media_5` |
| Vitórias | `V_media_3` | `V_media_5` |
| Aproveitamento | `Aproveitamento_media_3` | `Aproveitamento_media_5` |

Cada feature é calculada com `shift(1).rolling(w, min_periods=1).mean()`, garantindo que a temporada **T** usa apenas dados de **T-1, T-2, …** (sem data leakage).

In [10]:
# Tabela resumo das 15 features
import pandas as pd
descricao_features = pd.DataFrame({
    'Feature': FEATURES,
    'Tipo': (['Elenco'] * 3) + (['Janela Deslizante'] * 12),
})
print(f"Total de features: {len(FEATURES)}")
descricao_features

Total de features: 15


,Feature,Tipo
0,Plantel,Elenco
1,Estrangeiros,Elenco
2,Valor de Mercado Total,Elenco
3,Pts_media_3,Janela Deslizante
4,Pts_media_5,Janela Deslizante
5,SG_media_3,Janela Deslizante
6,SG_media_5,Janela Deslizante
7,Gols_Pro_media_3,Janela Deslizante
8,Gols_Pro_media_5,Janela Deslizante
9,Gols_Contra_media_3,Janela Deslizante
